In [1]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
target1 = 'Target1'
target2 = 'Target2'
train = pd.read_parquet('train.parquet')
val = pd.read_parquet('val.parquet')
test = pd.read_parquet('test.parquet')

In [3]:
for col in train.select_dtypes(include='number').columns:
    train[col].fillna(train[col].median(), inplace=True)

for col in train.select_dtypes(include='object').columns:
    train[col].fillna(train[col].mode()[0], inplace=True)




for col in val.select_dtypes(include='number').columns:
    val[col].fillna(val[col].median(), inplace=True)

for col in val.select_dtypes(include='object').columns:
    val[col].fillna(val[col].mode()[0], inplace=True)



for col in test.select_dtypes(include='number').columns:
    test[col].fillna(test[col].median(), inplace=True)

for col in test.select_dtypes(include='object').columns:
    test[col].fillna(test[col].mode()[0], inplace=True)

    


train_new = pd.get_dummies(train, drop_first=True)
val_new = pd.get_dummies(val, drop_first=True)
test_new = pd.get_dummies(test, drop_first=True)

/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_9702/2452799383.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(train[col].median(), inplace=True)
/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_9702/2452799383.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values al

In [4]:
dtrain = xgb.DMatrix(train_new.drop(columns=[target1, target2]), label=train_new[target2])
dval = xgb.DMatrix(val_new.drop(columns=[target1, target2]), label=val_new[target2])
dtest = xgb.DMatrix(test_new.drop(columns=[target1, target2]), label=test_new[target2])

# 4. Параметры модели
params = {
    'objective': 'binary:logistic',
    'eta': 0.1,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

evals = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(params, 
                  dtrain, 
                  num_boost_round=100, 
                  evals=evals,
                  early_stopping_rounds=10,
                  verbose_eval=10)

[0]	train-logloss:0.29206	val-logloss:0.29350
[10]	train-logloss:0.18577	val-logloss:0.18682
[20]	train-logloss:0.14133	val-logloss:0.14232
[30]	train-logloss:0.12605	val-logloss:0.12715
[40]	train-logloss:0.11282	val-logloss:0.11392
[50]	train-logloss:0.10410	val-logloss:0.10528
[60]	train-logloss:0.09949	val-logloss:0.10078
[70]	train-logloss:0.09527	val-logloss:0.09669
[80]	train-logloss:0.09363	val-logloss:0.09523
[90]	train-logloss:0.09154	val-logloss:0.09327
[99]	train-logloss:0.09000	val-logloss:0.09190


In [5]:
y_pred_prob = model.predict(dtest)
y_pred = (y_pred_prob > 0.5).astype(int)

def print_metrics(y_true, y_pred, y_pred_prob):
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_pred_prob))

print("\nTest set metrics:")
print_metrics(test_new[target2], y_pred, y_pred_prob)

print("\nTrain set metrics:")
y_train_pred_prob = model.predict(dtrain)
y_train_pred = (y_train_pred_prob > 0.5).astype(int)
print_metrics(train_new[target2], y_train_pred, y_train_pred_prob)

print("\nValidation set metrics:")
y_val_pred_prob = model.predict(dval)
y_val_pred = (y_val_pred_prob > 0.5).astype(int)
print_metrics(val_new[target2], y_val_pred, y_val_pred_prob)


Test set metrics:
Accuracy: 0.966706579534846
Precision: 0.8761987794245859
Recall: 0.7128502187019742
F1-score: 0.7861286747930383
ROC-AUC: 0.9813048720570687

Train set metrics:
Accuracy: 0.9666844246006578
Precision: 0.872887725167891
Recall: 0.7135156559651209
F1-score: 0.785196331795828
ROC-AUC: 0.9809585362119158

Validation set metrics:
Accuracy: 0.9656614035799813
Precision: 0.8692296595499135
Recall: 0.7086322474420793
F1-score: 0.7807580174927113
ROC-AUC: 0.9802425438332097
